In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
folder_path = "covtype.data"

col_names = [
    "Elevation", "Aspect", "Slope",
    "Horizontal_Distance_To_Hydrology", "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways", "Hillshade_9am", "Hillshade_Noon",
    "Hillshade_3pm", "Horizontal_Distance_To_Fire_Points",
    *[f"Wilderness_Area_{i}" for i in range(1, 5)],
    *[f"Soil_Type_{i}" for i in range(1, 41)],
    "Cover_Type"
]

rows = []
for filename in os.listdir(folder_path):
    name = os.path.splitext(filename)[0]
    values = name.split(",")
    if len(values) == len(col_names):
        rows.append(values)

df = pd.DataFrame(rows, columns=col_names)

pd.set_option('display.max_columns', None)

## WORKFLOW:
    1. Load data and quick overview (along with fixing dtypes, droppping duplicates, ...)
    3. Split data (test/train/val)
    2. EDA (visualisations, checking distributions, outliers, skewness on train set)

## 1. Data Overview

In [10]:
df.head()

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Wilderness_Area_1,Wilderness_Area_2,Wilderness_Area_3,Wilderness_Area_4,Soil_Type_1,Soil_Type_2,Soil_Type_3,Soil_Type_4,Soil_Type_5,Soil_Type_6,Soil_Type_7,Soil_Type_8,Soil_Type_9,Soil_Type_10,Soil_Type_11,Soil_Type_12,Soil_Type_13,Soil_Type_14,Soil_Type_15,Soil_Type_16,Soil_Type_17,Soil_Type_18,Soil_Type_19,Soil_Type_20,Soil_Type_21,Soil_Type_22,Soil_Type_23,Soil_Type_24,Soil_Type_25,Soil_Type_26,Soil_Type_27,Soil_Type_28,Soil_Type_29,Soil_Type_30,Soil_Type_31,Soil_Type_32,Soil_Type_33,Soil_Type_34,Soil_Type_35,Soil_Type_36,Soil_Type_37,Soil_Type_38,Soil_Type_39,Soil_Type_40,Cover_Type
0,1863,37,17,120,18,90,217,202,115,769,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6
1,1874,18,14,0,0,90,208,209,135,793,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6
2,1879,28,19,30,12,95,209,196,117,778,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6
3,1888,33,22,150,46,108,209,185,103,735,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6
4,1889,28,22,150,23,120,205,185,108,759,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6


In [14]:
print("Shape of dataset:")
display(df.shape)
display(df.info())

Shape of dataset:


(91448, 55)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91448 entries, 0 to 91447
Data columns (total 55 columns):
 #   Column                              Non-Null Count  Dtype
---  ------                              --------------  -----
 0   Elevation                           91448 non-null  int64
 1   Aspect                              91448 non-null  int64
 2   Slope                               91448 non-null  int64
 3   Horizontal_Distance_To_Hydrology    91448 non-null  int64
 4   Vertical_Distance_To_Hydrology      91448 non-null  int64
 5   Horizontal_Distance_To_Roadways     91448 non-null  int64
 6   Hillshade_9am                       91448 non-null  int64
 7   Hillshade_Noon                      91448 non-null  int64
 8   Hillshade_3pm                       91448 non-null  int64
 9   Horizontal_Distance_To_Fire_Points  91448 non-null  int64
 10  Wilderness_Area_1                   91448 non-null  int64
 11  Wilderness_Area_2                   91448 non-null  int64
 12  Wild

None

In [15]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Elevation,91448.0,2861.970508,237.451094,1863.0,2723.0,2903.0,3008.0,3849.0
Aspect,91448.0,137.728644,103.618391,0.0,54.0,109.0,203.0,360.0
Slope,91448.0,11.804588,6.578827,0.0,7.0,11.0,15.0,61.0
Horizontal_Distance_To_Hydrology,91448.0,260.540340,205.791256,0.0,95.0,212.0,376.0,1343.0
Vertical_Distance_To_Hydrology,91448.0,35.634481,43.621537,-146.0,6.0,23.0,53.0,554.0
Horizontal_Distance_To_Roadways,91448.0,3370.191289,1787.203039,0.0,1770.0,3451.0,4884.0,7117.0
Hillshade_9am,91448.0,218.216648,21.034790,0.0,208.0,222.0,232.0,254.0
Hillshade_Noon,91448.0,225.424110,16.783223,99.0,217.0,227.0,237.0,254.0
Hillshade_3pm,91448.0,139.264347,31.506156,0.0,121.0,140.0,159.0,248.0
Horizontal_Distance_To_Fire_Points,91448.0,3658.171409,1819.015793,0.0,2145.0,3780.0,5221.0,7173.0


In [16]:
df.nunique()

Elevation                             1665
Aspect                                 361
Slope                                   60
Horizontal_Distance_To_Hydrology       435
Vertical_Distance_To_Hydrology         440
Horizontal_Distance_To_Roadways       5784
Hillshade_9am                          177
Hillshade_Noon                         143
Hillshade_3pm                          247
Horizontal_Distance_To_Fire_Points    5827
Wilderness_Area_1                        2
Wilderness_Area_2                        2
Wilderness_Area_3                        2
Wilderness_Area_4                        2
Soil_Type_1                              2
Soil_Type_2                              2
Soil_Type_3                              2
Soil_Type_4                              2
Soil_Type_5                              2
Soil_Type_6                              2
Soil_Type_7                              2
Soil_Type_8                              2
Soil_Type_9                              2
Soil_Type_1

In [17]:
duplicates = df[df.duplicated()]
print(duplicates)

Empty DataFrame
Columns: [Elevation, Aspect, Slope, Horizontal_Distance_To_Hydrology, Vertical_Distance_To_Hydrology, Horizontal_Distance_To_Roadways, Hillshade_9am, Hillshade_Noon, Hillshade_3pm, Horizontal_Distance_To_Fire_Points, Wilderness_Area_1, Wilderness_Area_2, Wilderness_Area_3, Wilderness_Area_4, Soil_Type_1, Soil_Type_2, Soil_Type_3, Soil_Type_4, Soil_Type_5, Soil_Type_6, Soil_Type_7, Soil_Type_8, Soil_Type_9, Soil_Type_10, Soil_Type_11, Soil_Type_12, Soil_Type_13, Soil_Type_14, Soil_Type_15, Soil_Type_16, Soil_Type_17, Soil_Type_18, Soil_Type_19, Soil_Type_20, Soil_Type_21, Soil_Type_22, Soil_Type_23, Soil_Type_24, Soil_Type_25, Soil_Type_26, Soil_Type_27, Soil_Type_28, Soil_Type_29, Soil_Type_30, Soil_Type_31, Soil_Type_32, Soil_Type_33, Soil_Type_34, Soil_Type_35, Soil_Type_36, Soil_Type_37, Soil_Type_38, Soil_Type_39, Soil_Type_40, Cover_Type]
Index: []


In [ ]:
#fix data types
df = df.apply(pd.to_numeric)